<a href="https://colab.research.google.com/github/alibaba35t/ViewCountPredictor/blob/main/view_count_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import os


In [ ]:
df = pd.read_csv("/content/Instagram_Analytics_organized.csv")
df = df.drop(columns=df.columns[df.columns.str.contains('^Unnamed')])
X = df.drop(columns="target").astype(np.float32)
y = df["target"].astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train_scaled)
y_train = torch.FloatTensor(y_train.values).unsqueeze(1)

X_test = torch.FloatTensor(X_test_scaled)
y_test = torch.FloatTensor(y_test.values).unsqueeze(1)


train_dataset = TensorDataset(X_train,y_train)
train_loader = DataLoader(train_dataset, batch_size=64,shuffle=True)

input_dimension = X_train.shape[1]

print(df["target"].value_counts())

target
1    15000
0    14999
Name: count, dtype: int64


In [ ]:
class ViewPredictorModel(nn.Module):
  def __init__(self,input_dimension):
    super(ViewPredictorModel, self).__init__()
    self.layer_1 = nn.Linear(input_dimension,128)
    self.bn1 = nn.BatchNorm1d(128)

    self.layer_2 = nn.Linear(128,64)
    self.bn2 = nn.BatchNorm1d(64)

    self.layer_3 = nn.Linear(64,1)

    self.relu = nn.ReLU()
    self.sigmoid = nn.Sigmoid()
    self.dropout = nn.Dropout(p = 0.2)

  def forward(self, x):
    x = self.relu(self.bn1(self.layer_1(x)))
    x = self.dropout(x)
    x = self.relu(self.bn2(self.layer_2(x)))
    x = self.dropout(x)
    x = self.sigmoid(self.layer_3(x))
    return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model = ViewPredictorModel(input_dimension=input_dimension).to(device)



criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
os.makedirs("model_checkpoints_3", exist_ok=True)

In [ ]:
epochs = 100

for epoch in range(epochs):
  model.train()
  epoch_loss = 0
  for batch_X, batch_y in train_loader:

        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

  if (epoch + 1) % 5 == 0 or epoch == 0:
      avg_loss = epoch_loss / len(train_loader)
      print(f"loss:{avg_loss}")

      checkpoint_path = f"model_checkpoints_3/model_epoch_{epoch+1:02d}.pt"

      torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
      }, checkpoint_path)


loss:0.650146677017212
loss:0.6492575826644897
loss:0.6474490809440613
loss:0.6484642604192098
loss:0.6474781360626221
loss:0.6459100241661072
loss:0.6464622437159221
loss:0.6413061868349711
loss:0.6429289662043254
loss:0.6409912565549215
loss:0.6400692955652872
loss:0.639261327902476
loss:0.6387525143623352
loss:0.6373477017084758
loss:0.637503468990326
loss:0.6367658157348632
loss:0.6353072468439738
loss:0.6347220101356507
loss:0.6343575860659282
loss:0.6318926356633504
loss:0.6348659248352051


In [ ]:
loaded_model = ViewPredictorModel(input_dimension=X_test.shape[1]).to(device)


checkpoint_path = "model_checkpoints_3/model_epoch_90.pt"


checkpoint = torch.load(checkpoint_path, map_location=device)


loaded_model.load_state_dict(checkpoint['model_state_dict'])


loaded_model.eval()

X_test_tensor = X_test.to(device)
y_test_tensor = y_test.to(device)

with torch.no_grad():

    raw_predictions = loaded_model(X_test_tensor)

    predicted_classes = (raw_predictions >= 0.5).float()

    correct_predictions = (predicted_classes == y_test_tensor).sum().item()
    total_samples = y_test_tensor.size(0)

    accuracy = (correct_predictions / total_samples) * 100

print("=========================================")
print(f"{accuracy:.2f}")
print("=========================================")

🎯 Modelin Test Seti Accuracy Skoru: %50.68
